# dc-thermal-fusion — Autoencoder Training
Train a PyTorch autoencoder on fused FLIR A310 + DS18B20 features.

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np, torch, torch.nn as nn, pandas as pd
import matplotlib.pyplot as plt
from fusion.anomaly_detector import ThermalAutoencoder, AnomalyDetector


In [ ]:
# Load simulated data (run run_simulator.py first)
df = pd.read_csv('../data/samples/fused_features.csv')
features = df.drop(columns=['timestamp']).values
mu, sigma = features.mean(0), features.std(0) + 1e-8
X = (features - mu) / sigma
print(f'Shape: {X.shape}')

In [ ]:
# HYPERPARAMETERS — edit freely
EPOCHS = 30
LATENT_DIM = 4
LR = 1e-3
BATCH = 32


In [ ]:
model = ThermalAutoencoder(input_dim=X.shape[1], latent_dim=LATENT_DIM)
opt = torch.optim.Adam(model.parameters(), lr=LR)
Xt = torch.tensor(X, dtype=torch.float32)
losses = []
for ep in range(EPOCHS):
    idx = torch.randperm(len(Xt))
    for i in range(0, len(Xt), BATCH):
        b = Xt[idx[i:i+BATCH]]
        opt.zero_grad(); l = nn.functional.mse_loss(model(b), b); l.backward(); opt.step()
    losses.append(l.item())
    if (ep+1) % 5 == 0: print(f'Epoch {ep+1}/{EPOCHS}  loss={l.item():.5f}')

In [ ]:
plt.plot(losses); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.title('Reconstruction Loss'); plt.tight_layout(); plt.show()

In [ ]:
# Anomaly scoring
detector = AnomalyDetector(input_dim=X.shape[1], threshold=0.05)
scores = [detector.score(row) for row in X]
plt.hist(scores, bins=30); plt.axvline(0.05, color='r', label='threshold')
plt.xlabel('Recon MSE'); plt.legend(); plt.title('Anomaly Score Distribution'); plt.show()